# pandas Basics
## MAD 2502 — Applied Data

This notebook is a beginner-friendly introduction to **pandas**. It does not use the internet, so you can focus on learning how a `DataFrame` works.

### Learning goals
By the end, you should be able to:

- load a CSV file;
- inspect rows, columns, types, and missing values;
- select and filter data;
- sort rows;
- create new columns;
- compute summaries with `mean`, `median`, `value_counts`, and `groupby`;
- make a simple plot;
- save a cleaned table.

> **Habit:** Before doing analysis, first ask: *What does one row represent? What does each column mean?*


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

print("pandas version:", pd.__version__)
print("working folder:", Path.cwd())


## 1. Load a small dataset

The file `data/campus_events.csv` is included with this tutorial. Each row represents one fictional campus event.

`pd.read_csv(...)` reads the file and returns a `DataFrame`.


In [ ]:
data_path = Path("applied_data/data/campus_events.csv")

# This second path makes the notebook easier to run if Jupyter starts
# inside the applied_data folder.
if not data_path.exists():
    data_path = Path("data/campus_events.csv")

events = pd.read_csv(data_path)
events


### First inspection

These four commands answer different questions:

- `head()` — What do the first few rows look like?
- `shape` — How many rows and columns are there?
- `columns` — What are the column names?
- `info()` — What types did pandas infer? Are values missing?


In [ ]:
print("shape:", events.shape)
print("columns:", list(events.columns))
display(events.head())
events.info()


### Try it

1. Change `head()` to `head(3)`.
2. What does `events.shape[0]` represent?
3. What does `events.shape[1]` represent?


## 2. Select columns

A single column is usually a `Series`. Several columns form another `DataFrame`.


In [ ]:
print(type(events["category"]))
display(events["category"].head())

display(events[["event_name", "category", "attendance"]].head())


## 3. Filter rows

Boolean conditions create a True/False value for every row. Use that condition inside square brackets to keep only matching rows.


In [ ]:
large_events = events[events["attendance"] >= 100]
large_events[["event_name", "attendance"]]


In [ ]:
free_outdoor = events[
    (events["cost"] == 0)
    & (events["location_type"] == "Outdoor")
]

free_outdoor[["event_name", "cost", "location_type"]]


### Predict before running

What would this select?

```python
events[(events["category"] == "Music") | (events["category"] == "Sports")]
```

Write your prediction, then run it.


In [ ]:
music_or_sports = events[
    (events["category"] == "Music")
    | (events["category"] == "Sports")
]
music_or_sports


## 4. Sort

`sort_values` changes the order of the rows. It does not change the underlying values.


In [ ]:
events.sort_values("attendance", ascending=False)[
    ["event_name", "attendance"]
].head(5)


## 5. Missing values

Real data are often incomplete. `isna()` detects missing values.


In [ ]:
print(events.isna().sum())


In [ ]:
# Example: replace missing ratings with the median of the known ratings.
rating_median = events["rating"].median()
events["rating_filled"] = events["rating"].fillna(rating_median)

events[["event_name", "rating", "rating_filled"]].head(8)


**Important:** Filling missing data is a modeling decision. Do not automatically replace every missing value with zero. Zero and unknown are not the same thing.


## 6. Create a new column

Suppose we want a rough measure of attendance relative to cost. We can create a new column from existing columns.


In [ ]:
events["attendance_per_dollar"] = events["attendance"] / (events["cost"] + 1)

events[
    ["event_name", "attendance", "cost", "attendance_per_dollar"]
].head()


## 7. Summaries

Start with simple questions.


In [ ]:
print("mean attendance:", events["attendance"].mean())
print("median attendance:", events["attendance"].median())
print("largest attendance:", events["attendance"].max())
print()
print("events by category:")
print(events["category"].value_counts())


### Grouped summaries

`groupby` answers questions like:

> What is the mean attendance **for each category**?


In [ ]:
category_summary = (
    events.groupby("category", as_index=False)
    .agg(
        number_of_events=("event_name", "count"),
        mean_attendance=("attendance", "mean"),
        median_cost=("cost", "median"),
    )
    .sort_values("mean_attendance", ascending=False)
)

category_summary


## 8. Make a plot

A useful graph should have a purpose, labels, and readable categories.


In [ ]:
category_plot = category_summary.sort_values("mean_attendance")

ax = category_plot.plot(
    x="category",
    y="mean_attendance",
    kind="barh",
    legend=False,
    figsize=(7, 4),
)

ax.set(
    xlabel="Mean attendance",
    ylabel="Event category",
    title="Mean attendance by event category",
)

plt.tight_layout()
plt.show()


## 9. Save your result

A cleaned or summarized file can become the input for the next stage of a project.


In [ ]:
output_path = Path("applied_data/data/category_summary.csv")
if not output_path.parent.exists():
    output_path = Path("data/category_summary.csv")

category_summary.to_csv(output_path, index=False)
print("saved:", output_path)


# Practice

Use `events` to answer each question.

1. Which event had the highest attendance?
2. How many events were free?
3. What was the median attendance for outdoor events?
4. Which category had the highest average rating?
5. Create a new table containing only events with attendance above the overall median.
6. Make one labeled graph that helps answer a question about the dataset.

The goal is not to memorize commands. The goal is to learn the pattern:

**inspect → ask → select → compute → check → communicate**
